**Tabela** | ecommerce_categorias |

**Destino atual** | dbo.squad2_ecommerce_categorias |

**Destino produção** | squad2.ecommerce_categorias |

**Schema** | id_categoria, nome_categoria, id_categoria_pai |

**Chave PK** | id_categoria |

**Nulos** | id_categoria_pai (12 nulos — categorias raiz) |

**Depende de** | feat_squad2_99_helpers |

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
import logging
logging.getLogger("azure").setLevel(logging.WARNING)

TABELA        = "ecommerce_categorias"
MODO_GRAVACAO = "overwrite"

inicio = log_inicio(f"feat_squad2_02_ingestao_{TABELA}")

In [0]:
try:
    snapshot_id = get_snapshot_mais_recente()
    log.info(f"Snapshot selecionado: {snapshot_id}")

    df = ler_parquet(snapshot_id, TABELA)
    log.info(f"Leitura OK → {df.count()} linhas | {len(df.columns)} colunas")

except Exception as e:
    log.error(f"Erro ao ler {TABELA}: {str(e)}")
    raise

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when

# Schema
log.info("Schema:")
df.printSchema()

# Amostra
display(df)

# Nulos
df_nulos = df.select([
    spark_sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
])

log.info("Nulos por coluna:")
display(df_nulos)

# Validação de negócio:
# id_categoria_pai = null → categoria raiz (esperado)
total             = df.count()
total_raiz        = df.filter(col("id_categoria_pai").isNull()).count()
total_subcategoria = df.filter(col("id_categoria_pai").isNotNull()).count()

log.info(f"Total registros  : {total}")
log.info(f"Categorias raiz  : {total_raiz}")
log.info(f"Subcategorias    : {total_subcategoria}")

In [0]:
print(f"SQL_SCHEMA  : {SQL_SCHEMA}")
print(f"SQL_PREFIX  : {SQL_PREFIX}")
print(f"TABELA      : {TABELA}")
print(f"Destino SQL : {get_destino_sql(TABELA)}")

In [0]:
import subprocess
result = subprocess.run(
    ["grep", "-n", "SQL_SCHEMA",
     "/Workspace/Users/lidiafmagalhaes@gmail.com/merca-data-platform/notebooks/utils/feat_squad2_99_helpers.ipynb"],
    capture_output=True, text=True
)
print(result.stdout)

In [0]:
try:
    sucesso = gravar_sql(df, TABELA, mode=MODO_GRAVACAO)

    if sucesso:
        log.info(f"Gravação OK → {get_destino_sql(TABELA)}")
    else:
        raise Exception("Falha na gravação")

except Exception as e:
    log.error(f"Erro ao gravar {TABELA}: {str(e)}")
    raise

In [0]:
try:
    df_sql = ler_sql(TABELA)
    total_sql = df_sql.count()

    log.info(f"Validação OK → {get_destino_sql(TABELA)}")
    log.info(f"Registros gravados: {total_sql}")

    # Compara origem x destino
    if total_sql == total:
        log.info(" Origem e destino com mesmo número de registros!")
    else:
        log.warning(
            f" Divergência: "
            f"origem={total} | destino={total_sql}"
        )

    display(df_sql)

except Exception as e:
    log.error(f"Erro na validação: {str(e)}")
    raise

In [0]:
log_fim(f"feat_squad2_02_ingestao_{TABELA}", inicio)